# Pipeline complet — exécution manuelle

Ce notebook déroule tout le pipeline du projet étape par étape, pour le tester/déboguer manuellement sans passer par le cron GitHub Actions ni HuggingFace Jobs :

1. Génération + nettoyage des données (`make_dataset.py`)
2. Données météo externes (avec repli hors ligne si pas d'accès réseau)
3. Feature engineering (démo isolée du `FeatureEngineer`)
4. Entraînement (`train.py`)
5. Inférence batch (`predict_model.py`)
6. Test de l'API FastAPI (in-process, sans lancer `uvicorn`)

À lancer avec le kernel Python de l'environnement du projet (`pip install -r requirements.txt` au préalable).

In [1]:
import os
from pathlib import Path

# Le notebook vit dans notebooks/ — on se replace à la racine du projet
# pour que tous les chemins relatifs de config.yaml (data/raw, models/...) soient corrects.
if Path.cwd().name == "notebooks":
    os.chdir("..")

print("Répertoire de travail :", Path.cwd())

Répertoire de travail : /home/ronanguilloueee/dossier-standard


## 1. Génération + nettoyage des données

Équivalent de `python -m src.data.make_dataset`, mais étape par étape pour inspecter chaque résultat intermédiaire.

In [2]:
from src.data.make_dataset import load_raw_data, clean_data, RAW_DATA_DIR

df_raw = load_raw_data(RAW_DATA_DIR)
print(f"{len(df_raw)} lignes brutes")
df_raw.head()

2026-07-16 12:25:27,443 [INFO] Chargement des données brutes depuis data/raw (dataset synthétique)


510 lignes brutes


,date,age,revenu,anciennete_mois,categorie,region,cible
0,2026-01-17,38.9,45690.0,7.0,A,Nord,0
1,2026-05-21,18.9,25814.0,141.0,A,Est,0
2,2026-04-29,22.4,NaN,138.0,A,Nord,0
3,2026-03-21,65.6,35891.0,86.0,B,Nord,0
4,2026-03-20,24.6,43378.0,160.0,C,Est,1


In [3]:
df_clean = clean_data(df_raw)
print(f"{len(df_clean)} lignes après nettoyage structurel")
df_clean.describe(include="all")

2026-07-16 12:25:58,891 [INFO] 10 doublons supprimés


500 lignes après nettoyage structurel


,date,age,revenu,anciennete_mois,categorie,region,cible
count,500,455.000000,454.000000,464.000000,475,500,500.000000
unique,NaN,NaN,NaN,NaN,3,4,NaN
top,NaN,NaN,NaN,NaN,A,Sud,NaN
freq,NaN,NaN,NaN,NaN,194,146,NaN
mean,2026-03-31 20:35:31.200000256,39.737363,34982.803965,123.125000,NaN,NaN,0.354000
min,2026-01-01 00:00:00,4.400000,2164.000000,1.000000,NaN,NaN,0.000000
25%,2026-02-17 00:00:00,31.650000,29458.750000,66.750000,NaN,NaN,0.000000
50%,2026-03-31 00:00:00,39.900000,35185.000000,121.500000,NaN,NaN,0.000000
75%,2026-05-16 06:00:00,47.450000,40549.500000,184.250000,NaN,NaN,1.000000
max,2026-06-30 00:00:00,74.900000,63610.000000,238.000000,NaN,NaN,1.000000


## 2. Données météo externes

Appelle la vraie API Open-Meteo (nécessite un accès réseau). Si l'appel échoue (pas de réseau, API indisponible), on retombe automatiquement sur des valeurs météo factices pour pouvoir continuer le notebook hors ligne.

In [4]:
from src.data.make_dataset import fetch_external_data, merge_external_data

try:
    external_df = fetch_external_data(df_clean)
    df_merged = merge_external_data(df_clean, external_df)
    print("Données météo récupérées via l'API Open-Meteo.")
except Exception as e:
    print(f"Appel API météo impossible ({e}) — utilisation de valeurs factices pour continuer hors ligne.")
    df_merged = df_clean.copy()
    df_merged["temperature_max"] = 20.0
    df_merged["temperature_min"] = 10.0
    df_merged["precipitation"] = 0.0

df_merged[["date", "region", "temperature_max", "temperature_min", "precipitation"]].head()

2026-07-16 12:26:12,924 [INFO] Récupération météo pour Nord (2026-01-01 → 2026-06-30)
2026-07-16 12:26:13,842 [INFO] Récupération météo pour Est (2026-01-01 → 2026-06-30)
2026-07-16 12:26:14,088 [INFO] Récupération météo pour Ouest (2026-01-01 → 2026-06-30)
2026-07-16 12:26:14,363 [INFO] Récupération météo pour Sud (2026-01-01 → 2026-06-30)
2026-07-16 12:26:14,777 [INFO] Found credentials in environment variables.
2026-07-16 12:26:14,742 [INFO] Fichier persisté sur s3://mon-projet-ml-data/external/meteo/2026/07/meteo_20260716.csv


Données météo récupérées via l'API Open-Meteo.


,date,region,temperature_max,temperature_min,precipitation
0,2026-01-17,Nord,10.8,5.3,1.5
1,2026-05-21,Est,23.2,10.9,0.0
2,2026-04-29,Nord,21.0,10.3,0.0
3,2026-03-21,Nord,13.1,3.1,0.0
4,2026-03-20,Est,15.3,2.5,0.0


## 3. Sauvegarde du dataset traité

In [5]:
from src.data.make_dataset import save_processed_data, PROCESSED_DATA_DIR

output_file = save_processed_data(df_merged, PROCESSED_DATA_DIR)
print(f"Sauvegardé dans {output_file}")

2026-07-16 12:27:39,099 [INFO] Fichier persisté sur s3://mon-projet-ml-data/processed/dataset_clean/2026/07/dataset_clean_20260716.csv


Sauvegardé dans data/processed/dataset_clean.csv


## 4. Feature engineering — démo isolée

Aperçu de ce que `FeatureEngineer` calcule, indépendamment du reste du pipeline sklearn.

In [6]:
from src.models.feature_engineering import FeatureEngineer

fe = FeatureEngineer()
demo = fe.fit_transform(df_merged.drop(columns=["cible"]).head())
demo[["age", "revenu", "revenu_par_age", "revenu_par_annee_anciennete", "tranche_age", "categorie_region"]]

,age,revenu,revenu_par_age,revenu_par_annee_anciennete,tranche_age,categorie_region
0,38.9,45690.0,1174.550129,78325.714286,26-40,A_Nord
1,18.9,25814.0,1365.820106,2196.936170,18-25,A_Est
2,22.4,NaN,NaN,NaN,18-25,A_Nord
3,65.6,35891.0,547.118902,5008.046512,56+,B_Nord
4,24.6,43378.0,1763.333333,3253.350000,18-25,C_Est


## 5. Entraînement

Construit le pipeline complet (`FeatureEngineer` + `ColumnTransformer` + `LogisticRegression`), l'entraîne, affiche les métriques.

In [7]:
from src.models.train import load_data, build_model, train as train_model

X, y = load_data()
model = build_model()
trained_model, metrics = train_model(model, X, y)

metrics

2026-07-16 12:28:44,940 [INFO] Accuracy (test) : 0.600
2026-07-16 12:28:44,951 [INFO] 
              precision    recall  f1-score   support

           0       0.64      0.88      0.74        65
           1       0.27      0.09      0.13        35

    accuracy                           0.60       100
   macro avg       0.46      0.48      0.44       100
weighted avg       0.51      0.60      0.53       100

2026-07-16 12:28:44,752 [INFO] Accuracy (CV, 5 folds) : 0.610 (+/- 0.031)


{'accuracy': 0.6,
 'cv_accuracy_mean': np.float64(0.61),
 'cv_accuracy_std': np.float64(0.031024184114977114),
 'precision_classe_1': 0.2727272727272727,
 'recall_classe_1': 0.08571428571428572,
 'f1_classe_1': 0.13043478260869565}

## 6. Sauvegarde locale du modèle

Nécessaire pour tester l'inférence et l'API en local (source `"local"`), sans dépendre de MLflow ou du Hub HuggingFace.

In [8]:
import joblib
from src.config import load_config

CONFIG = load_config()
model_dir = Path(CONFIG["paths"]["models"])
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / CONFIG["paths"]["model_filename"]
joblib.dump(trained_model, model_path)
print(f"Modèle sauvegardé dans {model_path}")

Modèle sauvegardé dans models/model.joblib


## 7. Inférence batch (`predict_model.py`)

Équivalent de `python -m src.models.predict_model --source local`, sur quelques observations du dataset.

In [9]:
from src.models.predict_model import load_model, predict

loaded_model = load_model(source="local")
sample = X.sample(5, random_state=1)
predictions = predict(loaded_model, sample)
predictions[["prediction", "probabilite"]]

,prediction,probabilite
304,0,0.306285
340,0,0.403239
47,0,0.276328
67,0,0.361554
479,0,0.427755


## 8. Test de l'API FastAPI (in-process)

Teste l'API sans lancer `uvicorn` séparément, via `TestClient` — utile pour valider rapidement les endpoints pendant le développement.

Pour un vrai serveur accessible depuis l'extérieur du notebook : `make api` dans un terminal, puis `make streamlit` dans un autre.

In [10]:
from fastapi.testclient import TestClient
from src.api import model_loader

# Force la source "local" pour ce test (le modèle vient d'être sauvegardé à l'étape 6)
model_loader.CONFIG["api"]["model_source"] = "local"

from src.api.main import app

client = TestClient(app)

print(client.get("/health").json())
print(client.get("/model/info").json())

/home/ronanguilloueee/Formation_fullstack_IA/M09-AI-Engineering/.ai_engineering/lib/python3.12/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
2026-07-16 12:30:02,118 [INFO] HTTP Request: GET http://testserver/health "HTTP/1.1 200 OK"
2026-07-16 12:30:02,125 [INFO] HTTP Request: GET http://testserver/model/info "HTTP/1.1 200 OK"


{'status': 'ok'}
{'loaded': False, 'source': None, 'loaded_at': None, 'detail': None}


In [11]:
payload = {
    "age": 35,
    "revenu": 42000,
    "anciennete_mois": 18,
    "categorie": "A",
    "region": "Nord",
    # "date" omis -> l'API utilise aujourd'hui et récupère la météo elle-même
}

response = client.post("/predict", json=payload)
print(response.status_code)
response.json()

2026-07-16 12:30:27,696 [INFO] Chargement du modèle (source=local)...
2026-07-16 12:30:27,726 [INFO] HTTP Request: POST http://testserver/predict "HTTP/1.1 200 OK"


200


{'prediction': 1, 'probabilite': 0.5054381524591862}

## Pour aller plus loin

- **Lancer les vrais services** : `make api` (port 8000, docs sur `/docs`) puis `make streamlit` (port 8501) dans deux terminaux séparés, ou `docker compose up api streamlit`.
- **Lancer la suite de tests complète** : `make test` (ou `pytest tests/`) — exclut par défaut les tests d'intégration réseau (`pytest tests/ -m integration` pour les inclure).
- **Déclencher l'entraînement complet en conditions réelles** (HuggingFace Jobs + MLflow) : `python trigger_job.py` (nécessite les secrets configurés, voir le README).